# 12.5 MCP（Model Context Protocol）

> 🕐 预估学习时间：30分钟

MCP 是 Anthropic 提出的开放协议，旨在标准化 LLM 应用与外部工具/数据源之间的通信方式。它正快速成为 Agent 工具调用的行业标准。

本节涵盖：
- MCP 协议架构
- Client-Server 通信模型
- Resources / Tools / Prompts 三大原语
- MCP 与 Function Calling 的对比
- 构建 MCP Server 示例

## 1. 为什么需要 MCP？

**传统 Function Calling 的问题**：
- 每个 LLM 提供商有自己的工具定义格式（OpenAI、Anthropic、Google 均不同）
- 工具实现与 LLM 调用紧耦合，难以复用
- 缺乏标准化的发现、认证、错误处理机制
- 每个应用需要单独集成每个工具

**MCP 的解决方案**：
- **统一协议**：一次实现，多 LLM 复用
- **客户端-服务器架构**：工具作为独立服务运行
- **动态发现**：Client 自动发现 Server 的能力
- **标准化**：认证、传输、错误处理统一规范

**类比**：
- MCP 之于 LLM 工具调用 = USB-C 之于外设连接
- 一根标准线连接所有工具

In [ ]:
import json
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from enum import Enum

class MCPPrimitive(Enum):
    RESOURCE = "resource"
    TOOL = "tool"
    PROMPT = "prompt"

@dataclass
class MCPServerCapability:
    type: MCPPrimitive
    name: str
    description: str
    input_schema: Dict[str, Any] = field(default_factory=dict)

@dataclass
class MCPResource:
    uri: str
    name: str
    description: str
    mime_type: str = "text/plain"

class MCPServer:
    def __init__(self, name: str, version: str = "1.0.0"):
        self.name = name
        self.version = version
        self.resources: Dict[str, MCPResource] = {}
        self.tools: Dict[str, MCPServerCapability] = {}
        self.prompts: Dict[str, MCPServerCapability] = {}

    def register_resource(self, resource: MCPResource, handler):
        self.resources[resource.uri] = resource
        setattr(self, f'_resource_{resource.uri.replace("://", "_").replace("/", "_")}', handler)

    def register_tool(self, capability: MCPServerCapability, handler):
        self.tools[capability.name] = capability
        setattr(self, f'_tool_{capability.name}', handler)

    def list_capabilities(self):
        return {
            "resources": [
                {"uri": r.uri, "name": r.name, "description": r.description}
                for r in self.resources.values()
            ],
            "tools": [
                {"name": t.name, "description": t.description, "inputSchema": t.input_schema}
                for t in self.tools.values()
            ],
            "prompts": [
                {"name": p.name, "description": p.description}
                for p in self.prompts.values()
            ],
        }

    def call_tool(self, tool_name: str, arguments: Dict[str, Any]):
        if tool_name not in self.tools:
            return {"error": f"Tool '{tool_name}' not found"}
        handler = getattr(self, f'_tool_{tool_name}', None)
        if handler is None:
            return {"error": f"Handler for '{tool_name}' not found"}
        try:
            result = handler(**arguments)
            return {"content": [{"type": "text", "text": str(result)}]}
        except Exception as e:
            return {"error": str(e), "isError": True}

class MCPClient:
    def __init__(self):
        self.connected_servers: Dict[str, MCPServer] = {}

    def connect(self, server: MCPServer):
        self.connected_servers[server.name] = server
        print(f'[MCP Client] Connected to server: {server.name} v{server.version}')

    def list_all_capabilities(self):
        all_caps = {}
        for name, server in self.connected_servers.items():
            all_caps[name] = server.list_capabilities()
        return all_caps

    def call_tool(self, server_name: str, tool_name: str, arguments: Dict[str, Any]):
        if server_name not in self.connected_servers:
            return {"error": f"Server '{server_name}' not connected"}
        return self.connected_servers[server_name].call_tool(tool_name, arguments)

weather_server = MCPServer("weather-service", "1.0.0")

weather_server.register_resource(
    MCPResource("weather://current/beijing", "Beijing Weather", "Current weather in Beijing"),
    lambda: {"temperature": 22, "condition": "Sunny", "humidity": 45}
)

weather_server.register_tool(
    MCPServerCapability(
        MCPPrimitive.TOOL, "get_forecast", "Get weather forecast for a city",
        {"type": "object", "properties": {
            "city": {"type": "string", "description": "City name"},
            "days": {"type": "integer", "description": "Number of days", "default": 3}
        }, "required": ["city"]}
    ),
    lambda city, days=3: f"Forecast for {city}: {days}-day outlook is partly cloudy, 18-25°C"
)

db_server = MCPServer("database-service", "1.0.0")
db_server.register_tool(
    MCPServerCapability(
        MCPPrimitive.TOOL, "query_database", "Execute a SQL query",
        {"type": "object", "properties": {
            "query": {"type": "string", "description": "SQL query to execute"}
        }, "required": ["query"]}
    ),
    lambda query: f"Query executed: '{query}' returned 42 rows"
)

client = MCPClient()
client.connect(weather_server)
client.connect(db_server)

print('\n=== MCP Capability Discovery ===')
caps = client.list_all_capabilities()
print(json.dumps(caps, indent=2))

print('\n=== MCP Tool Call ===')
result = client.call_tool("weather-service", "get_forecast", {"city": "Shanghai", "days": 5})
print(json.dumps(result, indent=2))

result2 = client.call_tool("database-service", "query_database", {"query": "SELECT * FROM users"})
print(json.dumps(result2, indent=2))

print(f'\nKey: MCP provides a uniform protocol for tool/resource discovery and invocation.')
print(f'Any LLM that speaks MCP can use any MCP server without custom integration.')

## 2. MCP 的三大原语

| 原语 | 用途 | 示例 |
|------|------|------|
| **Resources** | 暴露数据/文件内容 | 数据库表、文件系统、API 数据 |
| **Tools** | 执行操作/计算 | 查询天气、发送邮件、运行代码 |
| **Prompts** | 预定义的提示模板 | 代码审查模板、翻译模板 |

**Resources vs Tools**：
- Resources：只读数据访问（类似 REST GET）
- Tools：有副作用的操作（类似 REST POST/PUT）
- Prompts：帮助用户更好地使用 LLM

## 3. MCP vs 传统 Function Calling

| 维度 | 传统 Function Calling | MCP |
|------|----------------------|-----|
| 工具定义 | 每个 Provider 格式不同 | 统一 JSON Schema |
| 工具发现 | 手动注册 | 自动发现（ListCapabilities） |
| 工具实现 | 紧耦合在应用代码中 | 独立服务，松耦合 |
| 认证/鉴权 | 各工具独立实现 | 协议内置 |
| 传输层 | HTTP/直接调用 | stdio/HTTP SSE |
| 复用性 | 每个应用重新集成 | 一次实现，多应用复用 |
| 生态 | 碎片化 | 逐渐统一（Anthropic/OpenAI 支持） |

**MCP 的实际影响**：
- 开源社区涌现大量 MCP Server（GitHub、Slack、Database 等）
- Claude Desktop、Cursor 等产品已原生支持 MCP
- 未来所有主流 LLM 提供商预计都将支持 MCP

## 4. MCP 在实际 Agent 中的应用

**MCP + ReAct Agent 的完整工作流**：

```
用户查询 → LLM 推理 → 决定调用工具
                          ↓
                   MCP Client → 发现可用工具
                          ↓
                   选择合适的 MCP Server 和 Tool
                          ↓
                   MCP Server 执行 → 返回结果
                          ↓
                   LLM 整合结果 → 生成回复
```

**最佳实践**：
- 每个 MCP Server 职责单一（如天气 Server、数据库 Server）
- 使用 MCP 的 Resources 暴露只读信息（如用户信息、配置）
- 使用 MCP 的 Tools 处理有副作用的操作（如发送消息、修改数据）
- 生产环境加认证层（OAuth2 / API Key）

In [ ]:
class MCPAgent:
    def __init__(self, mcp_client: MCPClient):
        self.client = mcp_client
        self.conversation_history = []

    def plan_and_execute(self, user_query: str):
        self.conversation_history.append({"role": "user", "content": user_query})
        caps = self.client.list_all_capabilities()
        available_tools = []
        for server_name, server_caps in caps.items():
            for tool in server_caps.get("tools", []):
                available_tools.append({
                    "server": server_name,
                    "tool": tool["name"],
                    "description": tool["description"]
                })

        print(f'Available tools: {len(available_tools)}')
        for t in available_tools:
            print(f'  [{t["server"]}] {t["tool"]}: {t["description"]}')

        if "weather" in user_query.lower():
            result = self.client.call_tool(
                "weather-service", "get_forecast", {"city": "Beijing", "days": 3}
            )
            return result
        elif "data" in user_query.lower() or "query" in user_query.lower():
            result = self.client.call_tool(
                "database-service", "query_database", {"query": "SELECT * FROM recent_data"}
            )
            return result

        return {"content": [{"type": "text", "text": "No relevant tool found"}]}

agent = MCPAgent(client)

print('\n=== MCP Agent in Action ===')
result = agent.plan_and_execute("What's the weather like in Beijing?")
print(f'Agent response: {json.dumps(result, indent=2)}')

result2 = agent.plan_and_execute("Query the recent data")
print(f'Agent response: {json.dumps(result2, indent=2)}')

print(f'\nKey: MCP Agent discovers tools dynamically and routes queries to correct servers.')
print(f'This enables building modular, extensible Agent systems.')

## 📝 课后思考题

1. MCP 的 Resources 和 Tools 分别适合什么场景？什么情况下一个功能应该设计为 Resource 而非 Tool？
2. 如果多个 MCP Server 提供了同名但功能不同的工具，Client 应如何处理冲突？
3. MCP 的 stdio 传输和 HTTP SSE 传输各有什么优缺点？在生产环境如何选择？
4. 如何为 MCP Server 设计合适的权限模型？